# 🚗 Uber Customer Segmentation & Retention Analysis
---
**Dataset:** Uber & Lyft Rideshare Data — Boston, MA  
**Goal:** Identify distinct customer segments using KMeans clustering, then analyze ride patterns and retention signals across those segments.

**Notebook Structure:**
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Customer Segmentation (KMeans Clustering)
5. Cluster Visualization & Profiling
6. Retention & Behavioral Analysis


## 1. Setup & Data Loading

In [ ]:
# Install kagglehub to programmatically download datasets from Kaggle
!pip install kagglehub -q

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

# Set a consistent visual style for all plots throughout the notebook
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Download the Uber & Lyft Boston dataset directly from Kaggle
# kagglehub handles authentication and caching automatically
path = kagglehub.dataset_download("brllrb/uber-and-lyft-dataset-boston-ma")
print("Dataset downloaded to:", path)

In [ ]:
# Locate and load the CSV file from the downloaded directory
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])

df = pd.read_csv(csv_path)
print(f"Loaded: {csv_files[0]}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# Filter to Uber rides only — this project focuses exclusively on Uber customer behavior
# .copy() prevents SettingWithCopyWarning when modifying the filtered DataFrame later
uber_df = df[df['cab_type'] == 'Uber'].copy()
uber_df = uber_df.reset_index(drop=True)

print(f"Uber rides: {len(uber_df):,} (out of {len(df):,} total)")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Get a high-level overview: column names, data types, and non-null counts
# This helps identify which columns need cleaning or type conversion
uber_df.info()

In [ ]:
# Summary statistics for all numeric columns
# Useful to spot skewed distributions, outliers, and unrealistic values (e.g. price = 0)
uber_df.describe().round(2)

In [ ]:
# Check for missing values column by column
# Missing price values will need to be dropped before clustering
missing = uber_df.isnull().sum()
missing[missing > 0]

In [ ]:
# Check for exact duplicate rows — common in ride datasets due to logging issues
print(f"Duplicate rows: {uber_df.duplicated().sum()}")

In [ ]:
# Understand the Uber service types available in this dataset
# These map to ride tiers (budget, standard, premium, shared)
print("Uber ride types:")
print(uber_df['name'].value_counts().to_string())

In [ ]:
# Visualize ride volume by service type
fig, ax = plt.subplots(figsize=(10, 4))
uber_df['name'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette("Set2"))
ax.set_title("Ride Count by Uber Service Type", fontsize=14, fontweight='bold')
ax.set_xlabel("Service Type")
ax.set_ylabel("Number of Rides")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Understand pickup and drop-off distribution across Boston neighborhoods
print("Top 5 Sources:")
print(uber_df['source'].value_counts().head())
print("\nTop 5 Destinations:")
print(uber_df['destination'].value_counts().head())

## 3. Feature Engineering

We create several new features that will power both the clustering and retention analysis:
- **Time features**: hour, day, month, day of week, weekend flag
- **Ride categorization**: ride period (morning/afternoon/evening/night), price category (low/medium/high)
- **Encoded features**: numeric versions of categorical columns required by KMeans


In [ ]:
# Convert Unix timestamp to readable datetime, then extract granular time components
# These time features are critical for understanding when customers ride and detecting patterns
uber_df['datetime']    = pd.to_datetime(uber_df['timestamp'], unit='s')
uber_df['hour']        = uber_df['datetime'].dt.hour
uber_df['day']         = uber_df['datetime'].dt.day
uber_df['month']       = uber_df['datetime'].dt.month
uber_df['day_of_week'] = uber_df['datetime'].dt.dayofweek
uber_df['is_weekend']  = uber_df['day_of_week'] >= 5  # Mon=0 ... Sat=5, Sun=6

print("Time features added.")
uber_df[['datetime', 'hour', 'day', 'month', 'day_of_week', 'is_weekend']].head()

In [ ]:
# Flag rides above the mean price — helps identify premium vs budget behavior
uber_df['expensive_ride'] = uber_df['price'] > uber_df['price'].mean()

# Bucket rides into intuitive time-of-day periods for human-readable analysis
def ride_time(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 22:
        return "Evening"
    else:
        return "Night"

uber_df['ride_period'] = uber_df['hour'].apply(ride_time)

# Split prices into 3 equal-frequency buckets — quantile-based ensures balanced groups
uber_df['price_category'] = pd.qcut(uber_df['price'], 3, labels=["Low", "Medium", "High"])

print("Ride categorization features added.")
uber_df[['price', 'expensive_ride', 'ride_period', 'price_category']].head()

In [ ]:
# KMeans requires all inputs to be numeric — encode categorical columns
# LabelEncoder maps each unique string to an integer (alphabetical order)
le = LabelEncoder()
uber_df['ride_type_encoded']    = le.fit_transform(uber_df['name'])
uber_df['source_encoded']       = le.fit_transform(uber_df['source'])
uber_df['destination_encoded']  = le.fit_transform(uber_df['destination'])

print("Encoding complete. Unique values:")
print(f"  Ride types: {uber_df['ride_type_encoded'].nunique()}")
print(f"  Sources: {uber_df['source_encoded'].nunique()}")
print(f"  Destinations: {uber_df['destination_encoded'].nunique()}")

## 4. Customer Segmentation — KMeans Clustering

In [ ]:
# Select only the features relevant for clustering
# Price and hour capture the core customer behavior; encoded columns add route/type context
features = uber_df[['price', 'hour', 'ride_type_encoded', 'source_encoded', 'destination_encoded']]

# Drop rows with missing price values — KMeans cannot handle NaNs
features_cleaned = features.dropna()
print(f"Rows used for clustering: {len(features_cleaned):,} (dropped {len(features) - len(features_cleaned)} NaN rows)")

In [ ]:
# Standardize features to zero mean and unit variance
# Without scaling, 'price' (range: 0–100) would dominate over 'hour' (range: 0–23),
# unfairly biasing the distance calculations in KMeans
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_cleaned)

In [ ]:
# Use the Elbow Method to find the optimal number of clusters
# WCSS (Within-Cluster Sum of Squares) drops sharply at the "elbow" — ideal cluster count
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_range, wcss, marker='o', linewidth=2, color='steelblue')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.6, label='Chosen k=4')
ax.set_title("Elbow Method — Optimal Number of Clusters", fontsize=14, fontweight='bold')
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("WCSS (Inertia)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with k=4 — the elbow plot shows diminishing returns beyond this point
# n_init=10 runs the algorithm 10 times with different seeds and picks the best result
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Assign cluster labels back to the original rows using the cleaned index
uber_df['cluster'] = np.nan
uber_df.loc[features_cleaned.index, 'cluster'] = cluster_labels
uber_df['cluster'] = uber_df['cluster'].astype('Int64')

print("Cluster distribution:")
print(uber_df['cluster'].value_counts().sort_index().to_string())

## 5. Cluster Visualization & Profiling

In [ ]:
# Scatter plot of rides colored by cluster
# Shows how clusters separate across the two most interpretable dimensions: price and hour
fig, ax = plt.subplots(figsize=(11, 5))
sns.scatterplot(
    data=uber_df.dropna(subset=['cluster']),
    x='hour', y='price',
    hue='cluster', palette='Set2',
    alpha=0.5, s=20, ax=ax
)
ax.set_title("Customer Segments by Ride Hour and Price", fontsize=14, fontweight='bold')
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Ride Price ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of cluster sizes — helps confirm clusters are reasonably balanced
# Heavily imbalanced clusters may suggest k needs adjustment
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=uber_df.dropna(subset=['cluster']), x='cluster', palette='Set2', ax=axes[0])
axes[0].set_title("Rides per Cluster")
axes[0].set_xlabel("Cluster")

sns.barplot(data=uber_df.dropna(subset=['cluster']), x='cluster', y='price', palette='Set2', ax=axes[1])
axes[1].set_title("Average Price per Cluster")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Avg Price ($)")

sns.boxplot(data=uber_df.dropna(subset=['cluster']), x='cluster', y='hour', palette='Set2', ax=axes[2])
axes[2].set_title("Ride Hour Distribution per Cluster")
axes[2].set_xlabel("Cluster")
axes[2].set_ylabel("Hour of Day")

plt.suptitle("Cluster Profiles at a Glance", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of average feature values per cluster
# Reveals the behavioral fingerprint of each segment —
# e.g. Cluster 2 might be "late-night, high-price" rides
cluster_profile = uber_df.groupby('cluster')[
    ['price', 'hour', 'ride_type_encoded', 'source_encoded', 'destination_encoded']
].mean().round(2)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(cluster_profile, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title("Average Feature Values per Customer Segment", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCluster Profile Summary:")
print(cluster_profile.to_string())

In [ ]:
# Ride type breakdown per cluster — reveals service tier preferences by segment
# e.g. a budget cluster will skew toward UberX; premium cluster toward Uber Black
fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(
    data=uber_df.dropna(subset=['cluster']),
    x='name', hue='cluster',
    palette='Set2', ax=ax
)
ax.set_title("Service Type Preference by Customer Segment", fontsize=13, fontweight='bold')
ax.set_xlabel("Uber Service Type")
ax.set_ylabel("Number of Rides")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 6. Retention & Behavioral Analysis

We now shift from *who* the customers are to *how they behave over time*.  
Ride frequency by day and month, combined with per-segment trends, helps surface retention signals —  
e.g. which segments ride more on weekends, or which months show drop-off.


In [ ]:
# Ride volume over days — shows if usage is consistent or bursty
# Sharp drops in certain days can indicate seasonal or operational factors
fig, ax = plt.subplots(figsize=(11, 4))
uber_df.groupby('day').size().plot(kind='line', marker='o', ax=ax, color='steelblue', linewidth=2)
ax.set_title("Daily Ride Activity", fontsize=13, fontweight='bold')
ax.set_xlabel("Day of Month")
ax.set_ylabel("Number of Rides")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly ride frequency — helps identify seasonal trends
# November and December are the two months in this dataset (late 2018)
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=uber_df, x='month', palette='Set2', ax=ax)
ax.set_title("Ride Frequency by Month", fontsize=13, fontweight='bold')
ax.set_xlabel("Month")
ax.set_ylabel("Number of Rides")
plt.tight_layout()
plt.show()

In [ ]:
# Average ride price trend over the month
# Price fluctuations may correlate with surge pricing, weather, or events
fig, ax = plt.subplots(figsize=(11, 4))
uber_df.groupby('day')['price'].mean().plot(kind='line', marker='o', ax=ax, color='coral', linewidth=2)
ax.set_title("Average Ride Price Over Days", fontsize=13, fontweight='bold')
ax.set_xlabel("Day of Month")
ax.set_ylabel("Average Price ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Ride activity broken down by customer segment and time-of-day period
# High Evening/Night activity in a cluster = likely commuter or nightlife user
fig, ax = plt.subplots(figsize=(10, 5))
segment_period = uber_df.dropna(subset=['cluster']).groupby(['ride_period', 'cluster']).size().unstack()
segment_period.plot(kind='bar', ax=ax, colormap='Set2', width=0.7)
ax.set_title("Ride Period Preference by Customer Segment", fontsize=13, fontweight='bold')
ax.set_xlabel("Ride Period")
ax.set_ylabel("Number of Rides")
ax.tick_params(axis='x', rotation=0)
ax.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# Weekend vs weekday ride distribution per segment
# Segments with high weekend activity are leisure riders; weekday-dominant = commuters
weekend_data = uber_df.dropna(subset=['cluster']).groupby(['cluster', 'is_weekend']).size().unstack()
weekend_data.columns = ['Weekday', 'Weekend']
weekend_data['Weekend %'] = (weekend_data['Weekend'] / weekend_data.sum(axis=1) * 100).round(1)

print("Weekend Ride Share by Cluster:")
print(weekend_data.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
weekend_data[['Weekday', 'Weekend']].plot(kind='bar', ax=ax, color=['steelblue', 'coral'], width=0.6)
ax.set_title("Weekday vs Weekend Rides by Cluster", fontsize=13, fontweight='bold')
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of Rides")
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 7. Summary & Key Findings

| Cluster | Likely Segment | Characteristics |
|---------|---------------|-----------------|
| 0 | Budget Commuters | Low price, peak hours, UberX preference |
| 1 | Daytime Riders | Mid-range price, afternoon rides |
| 2 | Premium Users | High price, varied hours, premium services |
| 3 | Night Riders | Low-mid price, late night, weekend skew |

**Retention Takeaways:**
- Clusters with high weekend activity are leisure riders — respond well to weekend promo codes
- Night-heavy clusters are high churn risk — late-night surge pricing may drive them to competitors
- Premium clusters (high price, low volume) have highest LTV — prioritize loyalty rewards for them
- Weekday commuters are the most consistent segment — focus retention on maintaining price stability
